# LLM DSL translation

Anton Antonov   
April 2026

----

## Setup

In [1]:
use LLM::Resources;

use DSL::Examples;
use ML::NLPTemplateEngine;

use DSL::Translators;
use Data::Translators;

In [2]:
#sink my $llm-evaluator = llm-configuration('ollama', model => 'gemma3:12b');
#sink my $llm-evaluator = llm-configuration('ollama', model => 'gpt-oss:20b');
sink my $llm-evaluator = llm-configuration('ollama', model => 'qwen3.8:27b-mlx');
#sink my $llm-evaluator = llm-configuration('chatgpt', model => 'gpt-4.1-mini');

---

## Pre-processing

In [3]:
sink my $spec = q:to/END/;
make a brand new recommender with the data @dsData;
apply LSI functions IDF, None, Cosine; 
recommend by profile for passengerSex:male, and passengerClass:1st;
join across with @dsData on "id";
echo the pipeline value;
END

In [4]:
my @cmds = $spec.split(/ \v | [ \h+ ';' \h+ \v]/, :skip-empty)».trim;
deduce-type(@cmds)

Vector(Atom((Str)), 5)

In [5]:
.raku.say for @cmds

"make a brand new recommender with the data \@dsData;"
"apply LSI functions IDF, None, Cosine;"
"recommend by profile for passengerSex:male, and passengerClass:1st;"
"join across with \@dsData on \"id\";"
"echo the pipeline value;"


---

## Language and workflows

### DSL examples 

In [6]:
#%html
dsl-examples().map({ $_.key X $_.value.keys }).flat(1).map({ <language workflow> Z=> $_ })».Hash.sort.Array
==> to-dataset()
==> to-html(field-names => <language workflow>)

language,workflow
Python,LSAMon
Python,QRMon
Python,SMRMon
Python,pandas
R,DataReshaping
R,LSAMon
R,QRMon
R,SMRMon
Raku,DataReshaping
Raku,LSAMon


In [7]:
dsl-workflow-separators()
==> to-json(:pretty)

{
  "Python": {
    "SMRMon": "\n.",
    "LSAMon": "\n.",
    "DataReshaping": "\n",
    "pandas": "\n",
    "QRMon": "\n."
  },
  "Raku": {
    "SMRMon": "\n.",
    "LSAMon": "\n.",
    "DataReshaping": ";\n",
    "TriesWithFrequencies": ";\n",
    "QRMon": "\n."
  },
  "WL": {
    "SMRMon": "⟹\n",
    "TriesWithFrequencies": ";\n",
    "LSAMon": "⟹\n",
    "Tabular": ";\n",
    "DataReshaping": ";\n",
    "QRMon": "⟹\n"
  },
  "R": {
    "QRMon": "%>%\n",
    "DataReshaping": "%>%\n",
    "SMRMon": "%>%\n",
    "LSAMon": "%>%\n"
  }
}

In [8]:
dsl-workflow-separators()<Python><pandas>

### NLP template engine

----

## Translation examples

### Data wrangling (WL)

In [9]:
sink my $spec = q:to/END/;
use dfTitanic;
filter by "sex" is "male";
group by the column "class";
show counts
END

In [10]:
llm-dsl-translation($spec, sep => Whatever, workflow-spec => 'Tabular', lang => 'WL', :$llm-evaluator):!echo

obj = Tabular[dfTitanic];
obj = Select[obj, #["sex"] == "male"&];
obj = GroupBy[obj, #1["class"] &];;
obj = Echo[Length /@ obj, "counts:"]

In [11]:
llm-dsl-translation($spec, sep => WhateverCode, workflow-spec => 'Tabular', lang => 'WL', :$llm-evaluator):!echo

obj = Tabular[dfTitanic];
obj = Select[obj, #["sex"] == "male"&];
obj = GroupBy[obj, #1["class"] &];;
Echo[Length /@ obj, "counts:"]

### Recommendations

In [12]:
sink my $spec = q:to/END/;
make a brand new recommender with the data @dsData;
apply LSI functions IDF, None, Cosine; 
recommend by profile for passengerSex:male, and passengerClass:1st;
join across with @dsData on "id";
echo the pipeline value;
END

In [13]:
llm-dsl-translation($spec, :$llm-evaluator)

ML::SparseMatrixRecommender.new(@dsData)
.apply-term-weight-functions('IDF', 'None', 'Cosine')
.recommend-by-profile({'passengerSex'=>'male', 'passengerClass'=>'1st'})
.join-across(@dsData, on => 'id')
.echo-value()

In [14]:
llm-dsl-translation($spec, workflow-spec => 'SMRMon', lang => 'Raku', sep => WhateverCode, :$llm-evaluator)

ML::SparseMatrixRecommender.new(@dsData)
.apply-term-weight-functions('IDF', 'None', 'Cosine')
.recommend-by-profile({'passengerSex'=>'male', 'passengerClass'=>'1st'})
.join-across(@dsData, on => 'id')
.echo-value()

In [15]:
llm-dsl-translation($spec, workflow-spec => 'SMRMon', lang => 'WL', method => 'nlp-template-engine', :$llm-evaluator)

There is no template SMRMon for the language WL.